![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--pii-sanitization-for-production-agents--pii_sanitization_tutorial)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NirDiamant/agents-towards-production/blob/main/tutorials/pii-sanitization-for-production-agents/pii_sanitization_tutorial.ipynb)

# PII Sanitization for Production AI Agents

**The missing layer between user input and your LLM.**

Every production agent processes user text. That text frequently contains sensitive personal information — emails, phone numbers, national IDs, private keys, and financial data. Without a sanitization layer, this PII reaches your LLM provider unfiltered.

This tutorial shows you how to add a pre-LLM PII sanitization hook to any agent pipeline.


## The Problem

A typical agent pipeline today:

```
User: 'My email is john@example.com, call me at +1-555-867-5309'
         ↓
    Agent processes
         ↓
    LLM Provider ← raw PII arrives here
```

### Why this matters

- **GDPR Article 25**: requires data protection by design and by default -- sanitizing PII before it reaches a third-party LLM provider is one way to support this, not a specific mandated implementation
- **EU AI Act**: obligations phase in on a schedule -- GPAI provider duties from August 2025, most high-risk system obligations from August 2, 2026, with some Annex III high-risk categories extended to August 2027. None of these mandate a specific technical hook, but agent pipelines handling regulated data fall within scope
- **HIPAA**: patient data in healthcare agent pipelines must be protected
- **LGPD (Brazil) / CCPA (California)**: impose broadly similar data-protection obligations in their own jurisdictions -- not a global extension of GDPR

The fix is a sanitization hook before your LLM call:

```
User input → [PII Sanitizer] → Agent → LLM Provider
                  ↑
     PII removed before LLM sees it
```



## Three Approaches to PII Sanitization

| Approach | Coverage | Setup | Data leaves your infra? |
|----------|----------|-------|--------------------------|
| Regex patterns | ~70% | Zero | No — fully local |
| Local NLP (Presidio) | ~80% | Medium | No — fully local |
| Semantic API (TrustBoost) | ~95% | Minimal | **Yes** — see disclosure below |

**Start with regex.** It covers most structured PII (emails, SSNs, API keys, national IDs) with zero setup and zero data leaving your infrastructure. Reach for a hosted semantic API only when you hit contextual or multilingual PII that regex can't catch — and only after reading the disclosure in Approach 2 below.

We will implement regex and semantic API approaches with executable code, and describe the local NLP approach.

![Data flow comparison](assets/data-flow-comparison.svg)


In [ ]:
# Install dependencies (same package list as requirements.txt)
!pip install requests langchain langchain-core langchain-openai langgraph -q


## Approach 1: Regex-based sanitization (start here)

Fast, zero dependencies, fully local — nothing leaves your infrastructure. Misses contextual PII (a name or address with no fixed pattern). This is the right default for most pipelines; only move to Approach 2 if you hit PII this can't catch.


In [ ]:
import re

PII_PATTERNS = [
    (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', 'EMAIL'),
    (r'(?:\+?\d{1,2}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b', 'PHONE'),
    (r'\b\d{3}-\d{2}-\d{4}\b', 'SSN'),
    (r'sk-[a-zA-Z0-9]{32,}', 'API_KEY'),
]

def regex_sanitize(text: str) -> str:
    for pattern, label in PII_PATTERNS:
        text = re.sub(pattern, '[REDACTED]', text)
    return text

# Test
test_input = 'My email is john@example.com, call me at +1-555-867-5309, and my SSN is 123-45-6789'
print('Input: ', test_input)
print('Output:', regex_sanitize(test_input))

# Known limitation, stated explicitly rather than hidden: the PHONE
# pattern also matches any other 10-digit sequence (order numbers,
# tracking numbers) as a false positive, and still misses phone numbers
# with unusual formatting. This is an inherent limit of regex-based phone
# detection, not something a bigger pattern fixes -- production systems
# use a dedicated library (e.g. `phonenumbers`) or escalate to Approach 2
# for this reason.


## Approach 2: Semantic API sanitization with TrustBoost (opt-in, not the default)

Handles contextual PII, multilingual patterns, and returns structured metadata for audit trails -- at a real privacy cost the previous approach doesn't have.

**Full disclosure -- read this before using it**: this approach sends your **raw, unsanitized** text to TrustBoost (hosted on Render/AWS), which in turn sends it to OpenAI (GPT-4o-mini) for detection. Your text crosses your trust boundary twice before sanitization happens.

- **TrustBoost** does not store your raw text -- it retains only the sanitized output and metadata for audit ([PRIVACY.md](https://github.com/teodorofodocrispin-cmyk/TrustBoost-PII-Sanitizer/blob/main/PRIVACY.md)).
- **OpenAI**, by default, retains API prompts and responses in abuse-monitoring logs for up to 30 days, regardless of what TrustBoost does with the data on its own side -- unless the account has approved Zero Data Retention (ZDR) or Modified Abuse Monitoring (MAM). This tutorial makes no claim about which retention mode TrustBoost's OpenAI account uses.

Either way, this is not a local operation -- your text does leave your infrastructure.

**Do not use this approach if**: you're in a regulated on-premise environment (e.g. HIPAA with no third-party data-sharing allowed), or you cannot allow raw text to leave your infrastructure under any circumstance. Use Approach 1 or a local NER model (Presidio) instead.

**Setup:** No installation required. Use `tx_hash='TRIAL'` for 50 free sanitizations.



In [ ]:
import requests

KNOWN_RISK_CATEGORIES = {'CRITICAL', 'PRIVATE', 'SENSITIVE', 'CLEAN'}

def trustboost_sanitize(text: str, wallet_id: str = 'my-agent') -> dict:
    """
    Sanitize PII using TrustBoost semantic API.
    Returns sanitized text + safety score + entity list for audit.

    Fails closed: any network error, non-200 response, malformed/nested-null
    payload, or an unrecognized risk_category value returns
    risk_category='CRITICAL' and an empty sanitized string, so a downstream
    security gate blocks the input instead of crashing or silently passing
    raw text through.
    """
    def _closed():
        return {
            'sanitized': '',
            'safety_score': 0.0,
            'risk_category': 'CRITICAL',
            'entities': [],
            'quota_remaining': 0
        }

    try:
        response = requests.post(
            'https://api.trustboost.dev/sanitize',
            json={
                'text': text,
                'tx_hash': 'TRIAL',
                'wallet_address': wallet_id
            },
            timeout=30
        )
        response.raise_for_status()
        data = response.json().get('data')
    except (requests.RequestException, ValueError):
        return _closed()

    # 'data' can legitimately be None (e.g. {"data": null}) even on a 200 --
    # validate the shape before calling .get() on anything nested in it.
    if not isinstance(data, dict):
        return _closed()

    usage_metrics = data.get('usage_metrics')
    if not isinstance(usage_metrics, dict):
        usage_metrics = {}

    # Allowlist, not a denylist: any value that isn't one of the four
    # categories TrustBoost documents -- missing, None, empty, or garbage --
    # is treated as CRITICAL, never silently passed through as safe.
    risk_category = data.get('risk_category')
    if risk_category not in KNOWN_RISK_CATEGORIES:
        risk_category = 'CRITICAL'

    return {
        'sanitized': data.get('sanitized_content', ''),
        'safety_score': data.get('safety_score', 0.0),
        'risk_category': risk_category,
        'entities': data.get('entities', []),
        'quota_remaining': usage_metrics.get('quota_remaining', 0)
    }

# Test — English
result = trustboost_sanitize('My email is john@example.com and my SSN is 123-45-6789')
print('Sanitized:    ', result['sanitized'])
print('Safety score: ', result['safety_score'])
print('Risk category:', result['risk_category'])
print('Entities:     ', result['entities'])
print('Quota left:   ', result['quota_remaining'])


In [ ]:
# Test — LATAM Spanish (RFC, CURP, Cedula)
result_es = trustboost_sanitize(
    'Cliente: Juan Lopez, RFC: LOPJ850101ABC, Tel: 55-1234-5678, Email: juan@empresa.com.mx',
    wallet_id='my-agent'
)
print('Input:    Cliente: Juan Lopez, RFC: LOPJ850101ABC, Tel: 55-1234-5678')
print('Output:  ', result_es['sanitized'])
print('Entities:', result_es['entities'])

In [ ]:
# Test — Japanese (My Number)
result_jp = trustboost_sanitize(
    '田中太郎、マイナンバー：123456789012、電話：090-1234-5678',
    wallet_id='my-agent'
)
print('Output:', result_jp['sanitized'])

## Integration with LangChain

Adding sanitization as a preprocessing step before any LLM call.


In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

def _regex_sanitize_as_dict(text: str) -> dict:
    """Wraps regex_sanitize()'s bare string return in the same shape
    trustboost_sanitize() uses, so create_privacy_aware_agent() can accept
    either sanitizer interchangeably."""
    sanitized_text = regex_sanitize(text)
    return {
        'sanitized': sanitized_text,
        'safety_score': None,
        'risk_category': 'CLEAN' if sanitized_text == text else 'PRIVATE',
        'entities': [],
        'quota_remaining': None,
    }

def create_privacy_aware_agent(llm, sanitizer=_regex_sanitize_as_dict, wallet_id: str = 'my-agent'):
    """
    Wraps any LangChain LLM with a PII sanitization layer.
    PII is removed before the message reaches the model.

    `sanitizer` defaults to the local regex sanitizer -- no data leaves
    your infrastructure. Pass a TrustBoost-backed sanitizer explicitly to
    opt into the hosted semantic API instead -- read Approach 2's
    disclosure above before doing so.
    """
    def invoke_with_sanitization(user_input: str) -> str:
        # Step 1: Sanitize before LLM sees the input
        sanitized = sanitizer(user_input)
        
        print(f'[PII Guard] Removed {len(sanitized["entities"])} entities'
              f' | Score: {sanitized["safety_score"]}'
              f' | Risk: {sanitized["risk_category"]}')
        
        # Step 2: Send clean input to LLM
        response = llm.invoke([HumanMessage(content=sanitized['sanitized'])])
        return response.content
    
    return invoke_with_sanitization

# Usage example (requires OPENAI_API_KEY)
# llm = ChatOpenAI(model='gpt-4o-mini')
# agent = create_privacy_aware_agent(llm)  # local regex by default, nothing leaves your infra
# response = agent('Help me email john@company.com about project X')
#
# To opt into the hosted TrustBoost API instead (see the disclosure in Approach 2 above):
# agent = create_privacy_aware_agent(llm, sanitizer=lambda text: trustboost_sanitize(text, 'my-agent'))
print('Privacy-aware agent wrapper ready.')


## Integration with LangGraph

Adding sanitization as a dedicated node in your agent graph.


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

SAFE_TO_CONTINUE = {'CLEAN', 'SENSITIVE', 'PRIVATE'}

class AgentState(TypedDict):
    user_input: str
    sanitized_input: str
    safety_score: float
    risk_category: str
    llm_response: str

def sanitize_node(state: AgentState) -> AgentState:
    """Sanitization node -- runs before any LLM node."""
    result = trustboost_sanitize(state['user_input'])
    return {
        **state,
        'sanitized_input': result['sanitized'],
        'safety_score': result['safety_score'],
        'risk_category': result['risk_category']
    }

def should_block(state: AgentState) -> str:
    """Route to continue only for recognized-safe categories -- an
    allowlist, not a denylist. Anything else (PRIVATE, CRITICAL, an
    unrecognized value, or a missing/None risk_category) blocks. This
    matters because trustboost_sanitize() already fails closed to
    'CRITICAL' on error -- but a node that only checked `== 'CRITICAL'`
    here would silently let through any *other* unexpected value too.
    """
    if state.get('risk_category') in SAFE_TO_CONTINUE:
        return 'continue'
    return 'block'

def block_node(state: AgentState) -> AgentState:
    return {**state, 'llm_response': 'Request blocked: critical PII detected.'}

def llm_node(state: AgentState) -> AgentState:
    # Your LLM call goes here -- using sanitized_input, not user_input
    return {**state, 'llm_response': f'Processed: {state["sanitized_input"]}'}

# Build the graph
graph = StateGraph(AgentState)
graph.add_node('sanitize', sanitize_node)
graph.add_node('llm', llm_node)
graph.add_node('block', block_node)
graph.set_entry_point('sanitize')
graph.add_conditional_edges('sanitize', should_block, {'continue': 'llm', 'block': 'block'})
graph.add_edge('llm', END)
graph.add_edge('block', END)
app = graph.compile()

# Test
result = app.invoke({'user_input': 'Send 149 USDC from wallet ABC123 to john@example.com'})
print('Sanitized input:', result['sanitized_input'])
print('Risk category:  ', result['risk_category'])
print('LLM response:   ', result['llm_response'])


## Production Patterns

### Autonomous quota management

TrustBoost returns `quota_remaining` on every response. Use it for autonomous budget management:

```python
result = trustboost_sanitize(text)
if result['quota_remaining'] < 10:
    # Notify operator — quota running low
    notify_operator('TrustBoost quota low — renewal needed')
```

### Audit trail

Every sanitization is logged automatically with safety score, risk category, and entity count — no raw PII stored. Use the returned metadata for your own compliance records:

```python
audit_record = {
    'timestamp': datetime.utcnow().isoformat(),
    'safety_score': result['safety_score'],
    'risk_category': result['risk_category'],
    'entities_removed': len(result['entities']),
    'agent_id': wallet_id
}
```


## Summary

You have implemented a production-ready PII sanitization layer for your agent pipeline:

1. **Regex baseline** — zero dependencies, covers standard formats
2. **Semantic sanitization** — TrustBoost handles contextual and multilingual PII
3. **LangChain integration** — wrapper pattern for any LLM
4. **LangGraph integration** — dedicated sanitization node with routing
5. **Production patterns** — audit trails and quota management

## Resources

- TrustBoost GitHub: https://github.com/teodorofodocrispin-cmyk/TrustBoost-PII-Sanitizer
- Health check: https://api.trustboost.dev/health
- Free preview (no wallet): POST https://api.trustboost.dev/sanitize/preview
- ClawHub: https://clawhub.ai/teodorofodocrispin-cmyk/trustboost-pii-sanitizer
